
### 1. Object Detection

**Object detection** is a core computer vision task that goes beyond image classification. Instead of assigning a single label to an entire image, a detector must simultaneously answer two questions for every object present:

- **What** is it? → class label (e.g. "person", "car", "dog")
- **Where** is it? → bounding box coordinates `(x1, y1, x2, y2)`

Formally, given an input image $I$, the model produces a set of triplets:

$$\{(b_i,\ c_i,\ s_i)\}_{i=1}^{N}$$

where $b_i$ is the bounding box, $c_i$ is the predicted class, and $s_i \in [0,1]$ is the confidence score.

---

### 2. Deep Learning Detection Architectures

Modern detectors are broadly categorised into two families:

#### Two-Stage Detectors — Faster R-CNN
Introduced by Ren *et al.* (2015), **Faster R-CNN** separates detection into two stages:

1. **Region Proposal Network (RPN)**: A small fully-convolutional network that slides over the feature map and scores ~2 000 candidate "anchor" boxes as either *object* or *background*, and simultaneously regresses their coordinates.
2. **Detection Head**: Each surviving proposal is RoI-pooled to a fixed size, then classified by a fully-connected head that outputs the final class label and a refined bounding box.

Because proposals are generated from shared convolutional features, Faster R-CNN is much faster than its predecessors (R-CNN, Fast R-CNN) while retaining high accuracy.

#### Single-Stage Detectors — YOLO
**YOLO** (You Only Look Once) is a family of real-time single-stage detectors that predict bounding boxes and class probabilities directly from the full image in a single forward pass.  
The key design principle is dividing the image into a grid where each cell simultaneously predicts bounding boxes and class scores — no separate proposal stage — enabling extremely fast inference suitable for real-time applications.


### 3. Backbone & Feature Pyramid Network (FPN)

Both models in this lab use **ResNet-50** as the backbone — a 50-layer residual network pre-trained on ImageNet (~1.2 M images, 1 000 classes). The residual connections

$$\mathbf{y} = \mathcal{F}(\mathbf{x},\,\{W_i\}) + \mathbf{x}$$

allow very deep networks to train without vanishing gradients.

On top of ResNet-50 sits a **Feature Pyramid Network (FPN)**, which merges low-resolution, semantically rich features from deep layers with high-resolution, spatially precise features from shallow layers. This multi-scale representation enables the detector to handle objects of vastly different sizes.

---

### 4. Fine-Tuning Dataset — Car Plate Detection

The **Car Plate Detection Dataset** is a domain-specific benchmark for license plate localization:

- Hundreds of annotated vehicle images with license plate bounding boxes
- Bounding boxes stored in COCO JSON format (`_annotations.coco.json`)
- 1 foreground category labeled **`licence`** in the annotations
- Labels are shifted by **+1** so Faster R-CNN's reserved background class (0) is not overwritten

Fine-tuning a COCO-pre-trained model on this dataset lets us reuse powerful visual features already learned from millions of images and adapt the final detection head to license plate detection.



In [1]:
# Mount Google Drive (only needed in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Lab2-Part2/Task4')
print('Working directory:', os.getcwd())

---

---

### Part 1: Object Detection with Pre-trained Models

In this section, you will use a pre-trained **Faster R-CNN** model to perform object detection. Unlike image classification (which predicts a single label for the entire image), object detection identifies **multiple objects** in an image and draws **bounding boxes** around each detected object.


---

#### Step 1: Load Detection Model and Helper Functions

The code below loads the pre-trained Faster R-CNN model and defines helper functions for:
- Downloading and resizing images from URLs
- Drawing bounding boxes and labels on detected objects
- Displaying results

In [ ]:
# Import required libraries
import torch
import matplotlib.pyplot as plt
import tempfile
from urllib.request import urlopen
from io import BytesIO
import time
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps

# Import Faster R-CNN model and weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

# Set device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Function 1: Display image with matplotlib
def display_image(image):
    plt.figure(figsize=(20, 15))
    plt.grid(False)
    plt.imshow(image)
    plt.axis("off")

# Function 2: Download and resize image from URL
def download_and_resize_image(url, new_width=256, new_height=256, display=False):
    _, filename = tempfile.mkstemp(suffix=".jpg")
    response = urlopen(url)
    image_data = BytesIO(response.read())
    pil_image = Image.open(image_data).convert("RGB")
    pil_image = ImageOps.fit(pil_image, (new_width, new_height), method=Image.Resampling.LANCZOS)
    pil_image.save(filename, format="JPEG", quality=90)
    print(f"Image downloaded to {filename}.")
    if display:
        display_image(pil_image)
    return filename

# Function 3: Draw bounding boxes and labels on image
def draw_boxes_pil(pil_img, boxes, labels, scores, categories, score_thresh=0.3):
    draw = ImageDraw.Draw(pil_img)

    # Try to load a nice font, fall back to default if unavailable
    try:
        font = ImageFont.truetype("arial.ttf", 18)
    except:
        font = ImageFont.load_default()

    # Draw each detection above threshold
    for box, lab, sc in zip(boxes, labels, scores):
        if sc < score_thresh:
            continue

        # Draw bounding box
        x1, y1, x2, y2 = [float(v) for v in box]
        draw.rectangle([x1, y1, x2, y2], width=4)

        # Draw label with white background
        name = categories[int(lab)]
        text = f"{name}: {float(sc):.2f}"
        tw, th = draw.textbbox((0,0), text, font=font)[2:]
        draw.rectangle([x1, max(0, y1 - th - 4), x1 + tw + 4, y1], fill="white")
        draw.text((x1 + 2, max(0, y1 - th - 2)), text, fill="black", font=font)

    return pil_img

---

#### Step 2: Test Object Detection on a Sample Image

The cell below downloads a public image from **Open Images v4** dataset, saves it locally, and displays it for inspection.

In [ ]:
# Download and display a test image
image_url = "https://farm1.staticflickr.com/4032/4653948754_c0d768086b_o.jpg"
downloaded_image_path = download_and_resize_image(image_url, 1280, 856, True)

In [ ]:
import os
from PIL import Image


local_image_path = None  

if local_image_path is not None:
    if not os.path.exists(local_image_path):
        raise FileNotFoundError(f"Image not found: {local_image_path}")
    # Validate it can be opened as an image
    with Image.open(local_image_path) as _img:
        pass
    downloaded_image_path = local_image_path
    print(f"Using local image: {downloaded_image_path}")
else:
    print(f"Using downloaded image: {downloaded_image_path}")

---

#### Step 3: Load and Run Faster R-CNN Object Detection

Use **Faster R-CNN with ResNet50-FPN** trained on the COCO dataset. This model can detect 80 different object categories including people, vehicles, animals, and common objects.

Run the cell below to perform object detection on the downloaded image.



**Note:** For Question 1, you will compare this Faster R-CNN model with **YOLO** (a fast single-stage detector). Instructions for loading YOLO are provided in the next section.



**Load Faster R-CNN Model**

Load the pre-trained Faster R-CNN model with ResNet50-FPN backbone, trained on COCO dataset (80 object categories).

In [ ]:
# Load pre-trained Faster R-CNN model (trained on COCO dataset)
det_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn(weights=det_weights).to(device)
detector.eval()  # Set to evaluation mode

# Get preprocessing transform and class categories
det_preprocess = det_weights.transforms()
det_categories = det_weights.meta["categories"]

print("Faster R-CNN loaded successfully!")
print(f"Number of classes: {len(det_categories)}")

**Image Loading**

Define a simple function to load images and convert them to RGB format.

In [ ]:
# Function to load image from path
def load_img_pil(path):
    return Image.open(path).convert("RGB")

**Detection Function**

Define the main detection function that runs the model on an image, filters results by confidence threshold, draws bounding boxes, and displays the annotated image.

In [ ]:
# Main detection function
def run_detector(detector, path, score_thresh=0.3, allowed_classes=None):
    # Load and preprocess image
    pil_img = load_img_pil(path)
    img_tensor = det_preprocess(pil_img).to(device)  # [3, H, W]

    # Run inference and measure time
    start_time = time.time()
    with torch.no_grad():
        outputs = detector([img_tensor])[0]
    end_time = time.time()

    # Extract detection results
    boxes = outputs["boxes"].detach().cpu().numpy()
    labels = outputs["labels"].detach().cpu().numpy()
    scores = outputs["scores"].detach().cpu().numpy()

    # Filter by confidence threshold
    keep = scores >= score_thresh

    # Optionally keep only selected class names
    if allowed_classes is not None:
        allowed_classes = {name.lower() for name in allowed_classes}
        class_keep = np.array([det_categories[int(label)].lower() in allowed_classes for label in labels])
        keep = keep & class_keep

    kept_count = int(keep.sum())
    if allowed_classes is None:
        print(f"Found {kept_count} objects (score >= {score_thresh}).")
    else:
        classes_str = ", ".join(sorted(allowed_classes))
        print(f"Found {kept_count} detections in classes: {classes_str} (score >= {score_thresh}).")
    print(f"Inference time: {end_time - start_time:.4f} seconds")

    # Visualize detections
    vis_img = pil_img.copy()
    vis_img = draw_boxes_pil(
        vis_img,
        boxes[keep],
        labels[keep],
        scores[keep],
        det_categories,
        score_thresh=score_thresh,
    )
    display_image(vis_img)

**Run Detection**

Execute object detection on the test image and display only `chair` detections with confidence threshold 0.8 (80%).

In [ ]:
# Run Faster R-CNN detection on the test image and keep only chairs
run_detector(detector, downloaded_image_path, score_thresh=0.8, allowed_classes=["chair"])

**Load YOLO Model**

Load YOLO nano (pre-trained on COCO 80 classes) from the `ultralytics` library. Model weights (`yolov8n.pt`) download automatically on first run if not cached.


In [ ]:
!pip install ultralytics

In [ ]:
# Load YOLO (one-stage detector) from ultralytics
# Uncomment the line below to install if not already available:
# !pip install ultralytics

from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time

# Load YOLO nano model pre-trained on COCO (80 classes)
# Weights are downloaded automatically on first run
model_yolo = YOLO('yolov8n.pt')
print("YOLO loaded successfully!")

# Run inference on the downloaded test image
img_path = downloaded_image_path
img = Image.open(img_path).convert('RGB')
results = model_yolo(img_path, conf=0.6, verbose=False)
start_time = time.time()
results = model_yolo(img_path, conf=0.6, verbose=False)
elapsed = time.time() - start_time
print(f"Detection time: {elapsed:.3f} seconds")

# Visualize results
result = results[0]
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.imshow(img)

for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    score = float(box.conf[0])
    cls_id = int(box.cls[0])
    cls_name = model_yolo.names[cls_id]
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor='orange', facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(x1, y1, f'{cls_name}:{score:.2f}', color='white', fontsize=8,
            bbox=dict(facecolor='orange', alpha=0.5))

plt.axis('off')
plt.title(f'YOLO Detections ({len(result.boxes)} objects found)')
plt.show()


In [ ]:
analysis = """
1. Inference time:
   - Faster R-CNN: 1.5496 seconds
   - YOLO: 0.029 seconds
   - Which is faster? YOLO

2. Detection accuracy:
   - Number of objects detected: Faster R-CNN 6 objects vs YOLO 4 objects
   - Missed objects or false detections: YOLO missed the chair on the far left and one chair behind the table. Faster R-CC detected all rocking chairs.

3. Trade-off analysis:
   - It is a classic Speed vs Accuracy trede-off. YOLO is much faster but less accurate while faster RNN is better at accuracy but slower.
"""
print(analysis)


---

### Part 2: Fine-tuning Faster R-CNN on Custom Dataset

In this section, you will **fine-tune** a pre-trained Faster R-CNN model on the
**Car Plate Detection Dataset**. This demonstrates how to adapt a
pre-trained model to a specific domain (detecting license plates in vehicle images)
rather than general object detection.

### Why Fine-tuning?

The Faster R-CNN model used in Part 1 was pre-trained on the **COCO dataset**, which contains 80 common object categories (people, cars, animals, etc.).  
However, **`licence` (the class label for license plates in this dataset)** is not one of the 80 COCO classes — the pre-trained model has never seen a license plate as a labelled target. If we run it directly on car plate images, it may occasionally detect the surrounding car as a "car", but it will never output a `licence` box because that category simply does not exist in its output head.

Fine-tuning solves this by:
1. **Keeping the backbone frozen (or lightly updated)** — the ResNet50-FPN backbone already extracts powerful low-level and mid-level features (edges, textures, shapes) that are useful for any vision task. Reusing these saves enormous compute and data.
2. **Replacing and retraining the classification head** — the final `FastRCNNPredictor` layer is swapped out for a new one with `num_classes = 2` (background + `licence`). Only this head (and optionally the RPN) needs to learn the new category from our domain-specific annotations.

This is the core idea of **transfer learning**: leverage knowledge already encoded in millions of pre-trained parameters, and only adapt the parts that are specific to your new task.

### About the Dataset

The **Car Plate Detection Dataset** contains:
- Annotated vehicle images with license plate bounding boxes
- Bounding boxes stored in `_annotations.coco.json` (COCO format: `[x, y, width, height]`)
- 1 foreground category labeled `licence` for license plate detection
- Labels are shifted by **+1** so Faster R-CNN's reserved background class (0) is not overwritten


In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

DATASET_ROOT = "Car Plate Detection"

# 1. Load the existing annotations JSON
with open(os.path.join(DATASET_ROOT, "_annotations.coco.json")) as f:
    annotation_data = json.load(f)

# 2. Automatically pick 3 annotated samples from current dataset
selected_filenames = [img["file_name"] for img in annotation_data["images"][:3]]
print(f"Detected and selected {len(selected_filenames)} images for visualization: {selected_filenames}")

# Build lookup: image_id -> annotations
ann_by_img = {}
for ann in annotation_data["annotations"]:
    ann_by_img.setdefault(ann["image_id"], []).append(ann)

# Build category id -> name lookup
cat_names = {cat["id"]: cat["name"] for cat in annotation_data["categories"]}

# Filter sampled image metadata based on the files we found
selected_samples = [img for img in annotation_data["images"] if img["file_name"] in selected_filenames]

fig, axes = plt.subplots(1, len(selected_samples), figsize=(6 * len(selected_samples), 5))
if len(selected_samples) == 1:
    axes = [axes]

for ax, img_meta in zip(axes, selected_samples):
    img_path = os.path.join(DATASET_ROOT, "images", img_meta["file_name"])
    pil_img = Image.open(img_path).convert("RGB")
    ax.imshow(pil_img)
    ax.set_title(img_meta["file_name"], fontsize=9)
    ax.axis("off")

    for ann in ann_by_img.get(img_meta["id"], []):
        x, y, w, h = ann["bbox"]  # [x, y, width, height]
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
        ax.add_patch(rect)
        label_name = cat_names.get(ann["category_id"], str(ann["category_id"]))
        ax.text(x, y - 4, label_name, color="lime", fontsize=8,
                bbox=dict(boxstyle="square,pad=0.1", fc="black", alpha=0.5))

plt.suptitle("Ground-Truth Bounding Boxes from _annotations.coco.json", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 1: Import Required Libraries
# ============================================================
import os
import json
import numpy as np
from PIL import Image
import torch
import torch.utils.data as data
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ============================================================
# SECTION 2: Define Custom Dataset Class (COCO JSON-backed)
# ============================================================
class CarPlateDataset(data.Dataset):
    """
    Car plate object detection dataset.
    Bounding boxes and class labels are loaded from _annotations.coco.json (COCO format).
    Classes are inferred from COCO categories and shifted by +1 for Faster R-CNN background class.
    """

    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms

        # Load COCO JSON
        coco_path = os.path.join(root, "_annotations.coco.json")
        with open(coco_path, "r") as f:
            coco = json.load(f)

        # Build image-id -> metadata lookup
        self._img_meta = {img["id"]: img for img in coco["images"]}

        # Group annotations by image_id
        self._ann_by_img = {}
        for ann in coco["annotations"]:
            self._ann_by_img.setdefault(ann["image_id"], []).append(ann)


        valid_img_ids = []
        missing_count = 0

        for img_id in sorted(self._img_meta.keys()):
            file_name = self._img_meta[img_id]["file_name"]
            # Check both possible locations (images/ subfolder or root)
            path1 = os.path.join(self.root, "images", file_name)
            path2 = os.path.join(self.root, file_name)

            if os.path.exists(path1) or os.path.exists(path2):
                valid_img_ids.append(img_id)
            else:
                missing_count += 1


        # Ordered list of image IDs (sorted for reproducibility)
        # self._img_ids = sorted(self._img_meta.keys())
        self._img_ids = valid_img_ids

        # Create a mapping for category_ids: original_id -> new_id
        # FasterRCNN expects class 0 to be background, so shift labels by +1
        coco_cat_ids = sorted({cat["id"] for cat in coco.get("categories", [])})
        if not coco_cat_ids:
            coco_cat_ids = sorted({ann["category_id"] for ann in coco.get("annotations", [])})
        self.category_id_map = {cat_id: i + 1 for i, cat_id in enumerate(coco_cat_ids)}

    def __getitem__(self, idx):
        img_id = self._img_ids[idx]
        meta = self._img_meta[img_id]
        anns = self._ann_by_img.get(img_id, [])

        # Load image from images subfolder
        img_path = os.path.join(self.root, "images", meta["file_name"])
        img = Image.open(img_path).convert("RGB")

        # Bounding boxes from COCO JSON
        # COCO format: [x, y, width, height] -> convert to [xmin, ymin, xmax, ymax]
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w > 0 and h > 0:
                boxes.append([x, y, x + w, y + h])
                # Remap category_id to shift it by +1
                labels.append(self.category_id_map[ann["category_id"]])

        num_objs = len(boxes)
        if num_objs == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)
        image_id_tensor = torch.tensor([img_id])

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id_tensor,
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)
        return img, target

    def __len__(self):
        return len(self._img_ids)

# ============================================================
# SECTION 3: Define Data Augmentation Transforms
# ============================================================
class Compose:
    """Composes several transforms together"""
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class ToTensor:
    """Converts PIL image to tensor"""
    def __call__(self, image, target):
        return F.to_tensor(image), target

class RandomHorizontalFlip:
    """Randomly flips image and adjusts bounding boxes"""
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, image, target):
        if torch.rand(1) < self.p:
            # Flip image
            image = F.hflip(image)
            w = image.shape[-1]
            # Adjust bounding boxes (flip x-coordinates)
            boxes = target["boxes"]
            if len(boxes) > 0:
                boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                target["boxes"] = boxes
        return image, target

def get_transform(train):
    """Returns appropriate transforms for training or testing"""
    t = [ToTensor()]
    if train:
        t.append(RandomHorizontalFlip(0.5))  # 50% chance of horizontal flip
    return Compose(t)

# ============================================================
# SECTION 4: Load Dataset and Create Data Loaders
# ============================================================
def collate_fn(batch):
    """Custom collate function for variable-size batches"""
    return tuple(zip(*batch))

# Set dataset path (single COCO dataset folder)
dataset_root = "Car Plate Detection"

# Build dataset objects
dataset_train_all = CarPlateDataset(dataset_root, transforms=get_transform(train=True))
dataset_eval_all  = CarPlateDataset(dataset_root, transforms=get_transform(train=False))

# Split full dataset into train/val/test (70%/15%/15%) with fixed seed
torch.manual_seed(42)
indices = torch.randperm(len(dataset_train_all)).tolist()
train_end = int(0.70 * len(indices))
val_end = int(0.85 * len(indices))
train_indices = indices[:train_end]
val_indices   = indices[train_end:val_end]
test_indices  = indices[val_end:]

dataset_train = torch.utils.data.Subset(dataset_train_all, train_indices)
dataset_val   = torch.utils.data.Subset(dataset_eval_all,  val_indices)
dataset_test  = torch.utils.data.Subset(dataset_eval_all, test_indices)

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    dataset_train, batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn
)
val_loader = torch.utils.data.DataLoader(
    dataset_val, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn
)
test_loader = torch.utils.data.DataLoader(
    dataset_test, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn
)

print(
    f"Dataset loaded: {len(dataset_train)} train, {len(dataset_val)} val, {len(dataset_test)} test images"
)

# ============================================================
# SECTION 5: Modify Model for Custom Task
# ============================================================
def get_model(num_classes):
    """
    Load pre-trained Faster R-CNN and replace classification head.
    """
    # Load pre-trained model (COCO weights)
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

    # Replace the classification head to detect num_classes instead of 80 (COCO)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

# Setup device, model, optimizer, and scheduler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#TODO complete the following code to create a model with the correct number of classes

num_classes = len(dataset_train_all.category_id_map) + 1

model = get_model(num_classes=num_classes).to(device)
# Optimizer: SGD with momentum and weight decay
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Learning rate scheduler: reduce LR by 10x every 3 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
print(f"Model initialized on {device}")

# ============================================================
# SECTION 6: Training and Evaluation Functions
# ============================================================
def train_one_epoch(model, optimizer, loader, device):
    """Train for one epoch and return average loss"""
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        # Move data to device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass (model returns loss dict in training mode). Model computes losses internally and returns a dict of loss components.
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(1, len(loader))

def evaluate_loss(model, loader, device):
    """Compute average validation/test loss without updating parameters"""
    was_training = model.training
    model.train()
    total_loss = 0.0

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            total_loss += loss.item()

    if was_training:
        model.train()
    else:
        model.eval()

    return total_loss / max(1, len(loader))

# ============================================================
# SECTION 7: Training Loop
# ============================================================
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)

# Track baseline history for later analysis questions
history = {'loss': [], 'val_loss': []}

num_epochs = 10
for epoch in range(num_epochs):
    # Train for one epoch
    loss = train_one_epoch(model, optimizer, train_loader, device)

    # Update learning rate
    lr_scheduler.step()

    # Evaluate validation loss each epoch
    val_loss = evaluate_loss(model, val_loader, device)

    # Store metrics so history['loss'][-1] is available in Part 4 answers
    history['loss'].append(loss)
    history['val_loss'].append(val_loss)

    # Print progress
    print(f"Epoch {epoch+1}/{num_epochs} | loss={loss:.4f} | val_loss={val_loss:.4f}")

# Final evaluation on test split
final_test_loss = evaluate_loss(model, test_loader, device)
print("=" * 60)
print(f"TRAINING COMPLETE! Final test_loss={final_test_loss:.4f}")
print("=" * 60)


**Prepare Images for Ground Truth vs Prediction**

Select the 3 sample images used earlier and resolve their absolute paths, ready for the side-by-side ground truth and prediction display in the next cells.

In [ ]:
import os

# Safely resolve absolute paths only for files that exist
_compare_img_paths = []
for fname in selected_filenames:
    full_path = os.path.join(DATASET_ROOT, "images", fname)
    if os.path.exists(full_path):
        _compare_img_paths.append(os.path.abspath(full_path))
    else:
        print(f"Warning: Sample image {fname} not found in dataset images, skipping.")

print(f"\nImages selected for ground-truth/prediction comparison ({len(_compare_img_paths)} total):")
for p in _compare_img_paths:
    print("  ", p)

#### Ground Truth vs Prediction

The cell below displays the same sample images with a side-by-side comparison of:
- Ground truth bounding boxes
- Model predictions

In [ ]:
import os
import matplotlib.patches as patches
from torchvision.transforms import functional as TF

SCORE_THRESH = 0.7   # same threshold used in show_predictions above

for abs_path in _compare_img_paths:
    fname = os.path.basename(abs_path)
    pil_img = Image.open(abs_path).convert("RGB")

    gt_img_id = None
    for img in annotation_data["images"]:
        if img["file_name"] == fname:
            gt_img_id = img["id"]
            break

    gt_items = []
    if gt_img_id is not None:
        for ann in annotation_data["annotations"]:
            if ann["image_id"] == gt_img_id:
                x, y, w, h = ann["bbox"]
                gt_items.append((x, y, w, h, ann["category_id"]))

    was_training = model.training
    model.eval()
    ft_tensor = TF.to_tensor(pil_img).to(device)
    with torch.no_grad():
        ft_out = model([ft_tensor])[0]
    if was_training:
        model.train()

    ft_boxes = ft_out["boxes"].cpu()
    ft_scores = ft_out["scores"].cpu()
    ft_labels = ft_out["labels"].cpu()
    ft_mask = ft_scores >= SCORE_THRESH
    boxes_after = ft_boxes[ft_mask]
    scores_after = ft_scores[ft_mask]
    labels_after = ft_labels[ft_mask]

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    axes[0].imshow(pil_img)
    axes[0].set_title("Ground Truth", fontsize=10)
    axes[0].axis("off")
    for x, y, w, h, cat_id in gt_items:
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none")
        axes[0].add_patch(rect)
        label_name = cat_names.get(cat_id, str(cat_id))
        axes[0].text(
            x,
            max(0, y - 4),
            label_name,
            color="lime",
            fontsize=8,
            bbox=dict(boxstyle="square,pad=0.1", fc="black", alpha=0.5),
        )

    axes[1].imshow(pil_img)
    axes[1].set_title("Prediction", fontsize=10)
    axes[1].axis("off")
    for b, s, l in zip(boxes_after, scores_after, labels_after):
        x1, y1, x2, y2 = b.tolist()
        label_str = cat_names.get(int(l), "background")
        axes[1].add_patch(
            patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor="crimson",
                facecolor="none",
            )
        )
        axes[1].text(
            x1,
            max(0, y1 - 4),
            f"{label_str}: {float(s):.2f}",
            color="crimson",
            fontsize=8,
            bbox=dict(boxstyle="square,pad=0.1", fc="white", alpha=0.7),
        )

    fig.suptitle(fname, fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()

#### Quantitative Evaluation: AP@0.5

The side-by-side comparison above gives a qualitative impression of the fine-tuned model. To measure performance rigorously, we compute **AP@0.5** (Average Precision at IoU threshold 0.5) on the held-out test split.

**What AP@0.5 measures:**
- A predicted box is counted as a **True Positive** only when its IoU with the matched ground-truth box is ≥ 0.5.
- All predictions are ranked by confidence score. Precision and Recall are computed at each rank cutoff, forming a **Precision-Recall (P-R) curve**.
- **AP** is the area under the P-R curve (all-point interpolation). It combines both precision and recall into a single number in [0, 1]: the closer to 1, the better.

Since this dataset has exactly **one foreground class** (`licence`), AP@0.5 and mAP@0.5 are numerically identical.


In [ ]:
# Quantitative evaluation: AP@0.5 on the test split
# For a single-class dataset, AP == mAP@0.5
from torchvision.ops import box_iou
import numpy as np
import torch

@torch.no_grad()
def compute_ap50_single_class(model, data_loader, device, fg_label=1):
    model.eval()

    # Collect predictions and ground truth per image
    predictions = []
    gt_cache = {}
    num_gt = 0
    image_idx = 0

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):
            gt_boxes = tgt["boxes"].cpu()
            gt_labels = tgt["labels"].cpu()
            gt_boxes = gt_boxes[gt_labels == fg_label]
            gt_cache[image_idx] = {
                "boxes": gt_boxes,
                "matched": np.zeros(len(gt_boxes), dtype=bool),
            }
            num_gt += len(gt_boxes)

            pred_boxes = out["boxes"].detach().cpu()
            pred_scores = out["scores"].detach().cpu()
            pred_labels = out["labels"].detach().cpu()

            keep = pred_labels == fg_label
            pred_boxes = pred_boxes[keep]
            pred_scores = pred_scores[keep]

            if pred_boxes.numel() > 0:
                order = torch.argsort(pred_scores, descending=True)
                for idx in order.tolist():
                    predictions.append({
                        "image_idx": image_idx,
                        "score": float(pred_scores[idx]),
                        "box": pred_boxes[idx],
                    })

            image_idx += 1

    if num_gt == 0:
        return 0.0, np.array([]), np.array([])

    predictions.sort(key=lambda x: x["score"], reverse=True)
    tp = np.zeros(len(predictions), dtype=np.float32)
    fp = np.zeros(len(predictions), dtype=np.float32)

    for i, pred in enumerate(predictions):
        gt_info = gt_cache[pred["image_idx"]]
        gt_boxes = gt_info["boxes"]

        if len(gt_boxes) == 0:
            fp[i] = 1.0
            continue

        ious = box_iou(pred["box"].unsqueeze(0), gt_boxes).squeeze(0).numpy()
        best_idx = int(np.argmax(ious))
        best_iou = ious[best_idx]

        if best_iou >= 0.5 and not gt_info["matched"][best_idx]:
            tp[i] = 1.0
            gt_info["matched"][best_idx] = True
        else:
            fp[i] = 1.0

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)
    recalls = tp_cum / (num_gt + 1e-12)
    precisions = tp_cum / (tp_cum + fp_cum + 1e-12)

    # All-point interpolation AP
    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    change_idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[change_idx + 1] - mrec[change_idx]) * mpre[change_idx + 1]))

    return ap, precisions, recalls

ap50, precisions, recalls = compute_ap50_single_class(model, test_loader, device, fg_label=1)
print(f"AP@0.5 (license plate): {ap50:.4f}")
print(f"Single-class setting => AP == mAP@0.5: {ap50:.4f}")

#### Experiment: AP under Different IoU Thresholds

A single AP@0.5 score only tells us how the model performs at one specific strictness level. By sweeping the IoU threshold from **0.3 to 0.9**, we can diagnose the nature of the model's errors:

| IoU threshold | Strictness | What a drop here means |
|---|---|---|
| 0.3 | Loose | Even rough overlaps count; a low AP here means plates are **not detected at all** |
| 0.5 | Standard | The conventional PASCAL VOC threshold |
| 0.7 | Strict | Boxes must be fairly tight; a drop here hints at **localization imprecision** |
| 0.9 | Very strict | Near-perfect overlap required; very sensitive to box boundary accuracy |

A **sharp decline** of mAP from low to high IoU thresholds indicates the model finds most plates but does not localize them precisely. A **flat curve** that stays high means both detection and localization are strong.


In [ ]:
# Experiment: AP under different IoU thresholds (single-class detector)
from torchvision.ops import box_iou
import numpy as np
import matplotlib.pyplot as plt
import torch

@torch.no_grad()
def compute_ap_single_class(model, data_loader, device, fg_label=1, iou_thresh=0.5):
    model.eval()

    predictions = []
    gt_cache = {}
    num_gt = 0
    image_idx = 0

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):
            gt_boxes = tgt["boxes"].cpu()
            gt_labels = tgt["labels"].cpu()
            gt_boxes = gt_boxes[gt_labels == fg_label]

            gt_cache[image_idx] = {
                "boxes": gt_boxes,
                "matched": np.zeros(len(gt_boxes), dtype=bool),
            }
            num_gt += len(gt_boxes)

            pred_boxes = out["boxes"].detach().cpu()
            pred_scores = out["scores"].detach().cpu()
            pred_labels = out["labels"].detach().cpu()

            keep = pred_labels == fg_label
            pred_boxes = pred_boxes[keep]
            pred_scores = pred_scores[keep]

            if pred_boxes.numel() > 0:
                order = torch.argsort(pred_scores, descending=True)
                for idx in order.tolist():
                    predictions.append({
                        "image_idx": image_idx,
                        "score": float(pred_scores[idx]),
                        "box": pred_boxes[idx],
                    })

            image_idx += 1

    if num_gt == 0:
        return 0.0

    predictions.sort(key=lambda x: x["score"], reverse=True)
    tp = np.zeros(len(predictions), dtype=np.float32)
    fp = np.zeros(len(predictions), dtype=np.float32)

    for i, pred in enumerate(predictions):
        gt_info = gt_cache[pred["image_idx"]]
        gt_boxes = gt_info["boxes"]

        if len(gt_boxes) == 0:
            fp[i] = 1.0
            continue

        ious = box_iou(pred["box"].unsqueeze(0), gt_boxes).squeeze(0).numpy()
        best_idx = int(np.argmax(ious))
        best_iou = ious[best_idx]

        if best_iou >= iou_thresh and not gt_info["matched"][best_idx]:
            tp[i] = 1.0
            gt_info["matched"][best_idx] = True
        else:
            fp[i] = 1.0

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)
    recalls = tp_cum / (num_gt + 1e-12)
    precisions = tp_cum / (tp_cum + fp_cum + 1e-12)

    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    change_idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[change_idx + 1] - mrec[change_idx]) * mpre[change_idx + 1]))

    return ap

iou_thresholds = [0.3, 0.5, 0.7, 0.9]
ap_values = [compute_ap_single_class(model, test_loader, device, fg_label=1, iou_thresh=t) for t in iou_thresholds]

print("AP under different IoU thresholds:")
for t, ap in zip(iou_thresholds, ap_values):
    print(f"  IoU={t:.1f} -> AP={ap:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(iou_thresholds, ap_values, marker='o')
plt.xlabel("IoU threshold")
plt.ylabel("AP (single class)")
plt.title("AP vs IoU threshold")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
analysis = '''
1. IoU Threshold vs AP:
   With the IoU threshold going up, the AP is decreasing.

2. Error Diagnosis via AP Trend:
   While AP@0.3 = 0.9908 and AP@0.5 = 0.9124, AP collapses at higher threshold. It indicates localization errors, which boxes are detected but not strictly aligned.

3. Single-Class Metric Interpretation:
   In a signle class, mAP@0.5, mean of AP@0.5 for different class, is AP@0.5 since the m = 1.
'''
print(analysis)